# Dynamic TableでMERGEロジックを自分で書く——CUSTOM_INCREMENTALが拓く複雑変換の新境地 — 検証ノートブック

## このノートブックについて

Zenn 記事「[Dynamic TableでMERGEロジックを自分で書く——CUSTOM_INCREMENTALが拓く複雑変換の新境地](https://zenn.dev/gtk0326/articles/i47-feature-update-2026-05-26-custom-incremental)」のハンズオン検証コードです。

記事と同じ手順を自分の Snowflake 環境で再現できます。

---

> **注意**: 各セルは上から順番に実行してください。最後のクリーンアップセルを必ず実行し、検証用オブジェクトを削除してください。

## ステップ1: スキーマとソーステーブルを準備する


In [ ]:
CREATE OR REPLACE DATABASE dt_custom_demo;
CREATE OR REPLACE SCHEMA dt_custom_demo.main;
USE SCHEMA dt_custom_demo.main;

-- 売上ログ（ソーステーブル）
CREATE OR REPLACE TABLE sales_log (
  sale_id  INT,
  user_id  INT,
  amount   NUMBER(10,2),
  sold_at  TIMESTAMP
);

-- sales_log 用ストリーム（CUSTOM_INCREMENTAL で使用）
CREATE OR REPLACE STREAM sales_log_stream ON TABLE sales_log;

INSERT INTO sales_log VALUES
  (1, 1, 1000.00, CURRENT_TIMESTAMP()),
  (2, 2,  500.00, CURRENT_TIMESTAMP()),
  (3, 1, 2500.00, CURRENT_TIMESTAMP()),
  (4, 3,  800.00, CURRENT_TIMESTAMP());

## ステップ2: 集計クエリの Dynamic Table を作成する（→ FULL に解決）

`SUM` + `GROUP BY` のような集計クエリは、差分行だけを処理して正しい集計値を維持することができません。そのため Snowflake は `FULL` を選択し、リフレッシュのたびに全件を再計算します。


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE user_sales_summary
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO
AS
  SELECT
    user_id,
    SUM(amount)  AS total_amount,
    COUNT(*)     AS sale_count
  FROM sales_log
  GROUP BY user_id;

## ステップ3: フィルタークエリの Dynamic Table を作成する（→ INCREMENTAL に解決）

`WHERE` のみのシンプルなフィルタークエリは、ソーステーブルの差分行に同じ条件を適用するだけで結果を維持できます。そのため Snowflake は `INCREMENTAL` を選択し、変更行のみを処理します。


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE large_sales
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO
AS
  SELECT sale_id, user_id, amount, sold_at
  FROM sales_log
  WHERE amount >= 1000;

## ステップ4: REFRESH USING を持つ Dynamic Table を作成する（→ CUSTOM_INCREMENTAL に解決）

`REFRESH USING` 句を追加すると、AUTO モードはこれをユーザーが増分ロジックを明示的に定義したと判断し、`CUSTOM_INCREMENTAL` を選択します。


In [ ]:
CREATE OR REPLACE DYNAMIC TABLE enriched_sales
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
  REFRESH_MODE = AUTO      -- REFRESH USING があれば CUSTOM_INCREMENTAL に自動解決
  INITIALIZE = ON_CREATE
AS
  SELECT sale_id, user_id, amount, sold_at
  FROM sales_log
REFRESH USING (
  INSERT INTO SELF
    SELECT sale_id, user_id, amount, sold_at
    FROM sales_log_stream
    WHERE METADATA$ACTION = 'INSERT'
);

## ステップ5: SHOW DYNAMIC TABLES で3モードを一覧確認する


In [ ]:
SHOW DYNAMIC TABLES IN SCHEMA dt_custom_demo.main;

## ステップ6: 新データを追加してインクリメンタルリフレッシュを確認する


In [ ]:
-- 2件追加
INSERT INTO sales_log VALUES
  (5, 2, 1500.00, CURRENT_TIMESTAMP()),
  (6, 3,  300.00, CURRENT_TIMESTAMP());

ALTER DYNAMIC TABLE enriched_sales REFRESH;

-- amount >= 1000 の行のみが存在するか確認
SELECT * FROM large_sales ORDER BY sale_id;

## クリーンアップ

検証で作成したオブジェクトをすべて削除します。

> **必ず実行してください。** Dynamic Table などを残すとバックグラウンドでリフレッシュが継続しクレジットが消費されます。

In [ ]:
DROP SCHEMA IF EXISTS dt_custom_demo.main;
ALTER DYNAMIC TABLE user_sales_summary SUSPEND;
DROP DYNAMIC TABLE IF EXISTS user_sales_summary;
ALTER DYNAMIC TABLE large_sales SUSPEND;
DROP DYNAMIC TABLE IF EXISTS large_sales;
ALTER DYNAMIC TABLE enriched_sales SUSPEND;
DROP DYNAMIC TABLE IF EXISTS enriched_sales;
DROP TABLE IF EXISTS sales_log;
DROP STREAM IF EXISTS sales_log_stream;

-- データベースを削除（内包するオブジェクトもすべて削除）
DROP DATABASE IF EXISTS dt_custom_demo;